# palsynet on Colab: 3D CNN on 5 s video segments

Trains and evaluates `r3d_18` with subject-level 5-fold cross-validation, on **landmark-aligned faces** and on **raw frames**.

**Before running**
1. *Runtime → Change runtime type →* a **GPU** (T4 works; L4/A100 is faster).
2. Upload `dist/palsynet_colab.zip` from the project to **Google Drive → `MyDrive/palsynet/`**.
3. *Runtime → Run all.* Approve the Drive access prompt.

Results (metrics, predictions, `best.pt` per fold) are written to `MyDrive/palsynet/runs/`, so they survive a disconnect.
If the session drops, rerun all: finished folds are skipped (set `RESUME_*` below to the existing `cv_*` folder).

In [ ]:
import torch, os, subprocess
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> GPU, then rerun."
print(torch.__version__, torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.0f} GB")
print("CPUs:", os.cpu_count())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/palsynet'
ZIP = f'{DRIVE_DIR}/palsynet_colab.zip'
RUNS_DIR = f'{DRIVE_DIR}/runs'
assert os.path.exists(ZIP), f'Upload palsynet_colab.zip to {DRIVE_DIR}/ first'
os.makedirs(RUNS_DIR, exist_ok=True)

# Work on the VM's local disk (much faster than reading videos from Drive).
!rm -rf /content/palsynet && mkdir -p /content/palsynet && unzip -q "$ZIP" -d /content/palsynet
%cd /content/palsynet
!ls data/aligned/affected | wc -l && ls data/aligned/unaffected | wc -l

In [ ]:
# Raw videos for the raw-frame baseline (~250 MB from the Hugging Face Hub, pinned revision).
!python scripts/download.py

In [ ]:
# Settings
NUM_WORKERS = max(2, os.cpu_count())   # Colab free tier has 2 vCPUs; data decoding can be the bottleneck
EXTRA = f'--set data.num_workers={NUM_WORKERS}'
RESUME_ALIGNED = ''   # e.g. f'{RUNS_DIR}/cv_r3d_18_aligned_20261114-010203' to continue an interrupted run
RESUME_RAW = ''
ENV = f'PYTHON=python RUNS_DIR="{RUNS_DIR}"'

## Smoke test
A few training steps to catch problems and time a step before the long runs. Nothing is kept.

In [ ]:
!python scripts/train.py --config configs/r3d_18_aligned.yaml --set output_dir=/tmp/smoke train.epochs=1 data.eval_segments_per_video=1 data.num_workers=$NUM_WORKERS --max-steps 10 2>&1 | grep -E "params|device|epoch 0|Error"
import json, glob
h = json.load(open(sorted(glob.glob('/tmp/smoke/*/history.json'))[-1]))[0]
print(f"10 train steps + short val: {h['time_s']:.0f} s")
!rm -rf /tmp/smoke

## 1. r3d_18 on landmark-aligned faces (5 folds)

In [ ]:
!{ENV} CV_DIR="{RESUME_ALIGNED}" bash scripts/run_cv.sh configs/r3d_18_aligned.yaml {EXTRA} 2>&1 | grep -E --line-buffered "results ->|=====|^epoch|fold [0-9] test|folds,|metric|accuracy|balanced|f1|auc|confusion|misclass|  [a-z]+_[0-9]+ |Error|Traceback"

## 2. r3d_18 on raw frames (5 folds), the baseline

In [ ]:
!{ENV} CV_DIR="{RESUME_RAW}" bash scripts/run_cv.sh configs/r3d_18.yaml {EXTRA} 2>&1 | grep -E --line-buffered "results ->|=====|^epoch|fold [0-9] test|folds,|metric|accuracy|balanced|f1|auc|confusion|misclass|  [a-z]+_[0-9]+ |Error|Traceback"

## Compare

In [ ]:
import glob, pandas as pd, json
rows = []
for cv in sorted(glob.glob(f'{RUNS_DIR}/cv_r3d_18*')):
    for f in sorted(glob.glob(f'{cv}/*_fold*_*/test_metrics.json')):
        v = json.load(open(f))['video']
        rows.append({'run': os.path.basename(cv), 'fold': f.split('_fold')[1][0], **{k: v[k] for k in ['accuracy', 'balanced_accuracy', 'f1', 'auc']}})
df = pd.DataFrame(rows)
display(df.groupby('run')[['accuracy', 'balanced_accuracy', 'f1', 'auc']].agg(['mean', 'std']).round(3))
for s in sorted(glob.glob(f'{RUNS_DIR}/cv_r3d_18*/summary.txt')):
    print('\n#', s.split('/')[-2]); print(open(s).read())

## Other models (optional)
Same pattern with another config, e.g.
```
!{ENV} bash scripts/run_cv.sh configs/vivit_b16.yaml {EXTRA}
!{ENV} bash scripts/run_cv.sh configs/r2plus1d_18.yaml {EXTRA} --set data.batch_size=8 train.grad_accum=1
```
For ViViT or other models on aligned faces, add `--set raw_root=data/aligned`.